# 07 - Comprehensive Benchmark Analysis & Statistical Evaluation

This notebook performs the complete scientific evaluation pipeline across all target dimensions ($D \in \{2, 3, 5\}$), noise levels ($\sigma \in \{0.0, 0.05\}$), **4 Prompt Strategies** (`Baseline`, `Thinking`, `Vectorization`, `Guided`), and **Classical Baselines** (`CMA-ES`, `DE`, `PSO`).

### 🔬 3-Tier Hypothesis Testing Structure:
1. **Tier 1: Classical Baselines Comparison** (`CMA-ES` vs. `DE` vs. `PSO`).
2. **Tier 2: LLaMEA Prompt Strategies vs. Classical Baselines** (Assessing performance of each LLM strategy against classical optimizers).
3. **Tier 3: Prompt Strategy Ablation Matrix** (Direct head-to-head evaluation between prompt engineering paradigms).

In [1]:
# ── 1. Setup Environment, Paths & Styling ──────────────────────────────────────
import os
import io
import re
import sys
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import kruskal, mannwhitneyu

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, RESULTS_DIR

IOH_LOGS_DIR = DATA_DIR / 'ioh_logs'
FIGURES_DIR  = RESULTS_DIR / 'figures'
REPORTS_DIR  = RESULTS_DIR / 'reports'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

BBOB_NAMES = {
    1: 'Sphere (f1)',
    8: 'Rosenbrock (f8)',
    11: 'Discus (f11)',
    15: 'Rastrigin (f15)',
    21: 'Gallagher 101 Peaks (f21)'
}

BBOB_CLASSES = {
    1: 'Separable',
    8: 'Low Conditioning',
    11: 'High Conditioning',
    15: 'Multi-Modal (Global Structure)',
    21: 'Multi-Modal (Weak Structure)'
}

SOLVER_COLORS = {
    'CMAES': '#2C3E50',                  # Slate Black
    'DE': '#2980B9',                     # Royal Blue
    'PSO': '#D35400',                    # Dark Orange
    'LLaMEA (Baseline)': '#27AE60',      # Forest Green
    'LLaMEA (Thinking)': '#16A085',      # Deep Emerald
    'LLaMEA (Vectorization)': '#00BCD4', # Bright Cyan
    'LLaMEA (Guided)': '#8E44AD',        # Royal Purple
    'LLaMEA Champion': '#27AE60'         # Default Fallback Green
}

SOLVER_DASHES = {
    'CMAES': 'solid',
    'DE': 'solid',
    'PSO': 'solid',
    'LLaMEA (Baseline)': 'dash',
    'LLaMEA (Thinking)': 'solid',
    'LLaMEA (Vectorization)': 'dashdot',
    'LLaMEA (Guided)': 'solid',
    'LLaMEA Champion': 'solid'
}

print(f"IOH Logs Directory: {IOH_LOGS_DIR}")
print(f"Figures Output:     {FIGURES_DIR}")
print(f"Reports Output:     {REPORTS_DIR}")


IOH Logs Directory: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs
Figures Output:     /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures
Reports Output:     /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports


## 1. Helper Functions: IOH Log Parsing & Data Extraction

In [2]:
def parse_ioh_dat_file(dat_path: Path) -> list[pd.DataFrame]:
    """Parses a raw IOHprofiler .dat file into a list of DataFrames per run."""
    if not dat_path.exists():
        return []
        
    text = dat_path.read_text(encoding='utf-8', errors='ignore')
    run_chunks = re.split(r'\n(?=function evaluation\s+current best\s+)', text)
    if len(run_chunks) <= 1:
        run_chunks = text.split('"function evaluation"')
        
    runs = []
    for chunk in run_chunks:
        chunk = chunk.strip()
        if not chunk or ('function evaluation' in chunk and len(chunk.splitlines()) <= 1):
            continue
            
        lines = [line.strip() for line in chunk.splitlines() if line.strip() and not line.startswith(('"', '#'))]
        if not lines:
            continue
            
        csv_data = 'evaluations raw_y\n' + '\n'.join(lines)
        try:
            df = pd.read_csv(io.StringIO(csv_data), sep=r'\s+')
            df['evaluations'] = pd.to_numeric(df['evaluations'], errors='coerce')
            df['raw_y'] = pd.to_numeric(df['raw_y'], errors='coerce')
            df = df.dropna().reset_index(drop=True)
            if len(df) > 0:
                runs.append(df)
        except Exception:
            pass
            
    return runs


def load_problem_benchmark_data(problem_dir: Path) -> dict[str, list[pd.DataFrame]]:
    """Load all algorithm trajectories from a problem directory, standardized by display name."""
    data = {}
    if not problem_dir.exists():
        return data
        
    for algo_dir in sorted(problem_dir.iterdir()):
        if not algo_dir.is_dir():
            continue
            
        clean_name = algo_dir.name.lower()
        if any(dk in clean_name for dk in ['dummy-llm', 'failing-llm', 'local-model']):
            continue
            
        if 'cmaes' in clean_name:
            display_name = 'CMAES'
        elif clean_name in ['de', 'de_clean', 'de_noisy']:
            display_name = 'DE'
        elif 'pso' in clean_name:
            display_name = 'PSO'
        elif 'baseline' in clean_name:
            display_name = 'LLaMEA (Baseline)'
        elif 'thinking' in clean_name:
            display_name = 'LLaMEA (Thinking)'
        elif 'vectorization' in clean_name:
            display_name = 'LLaMEA (Vectorization)'
        elif 'guided' in clean_name:
            display_name = 'LLaMEA (Guided)'
        elif 'llamea_champion' in clean_name:
            display_name = 'LLaMEA Champion'
        else:
            display_name = clean_name.upper()
            
        runs = []
        for dat_file in algo_dir.rglob('*.dat'):
            runs.extend(parse_ioh_dat_file(dat_file))
                
        if runs:
            if display_name in data:
                data[display_name].extend(runs)
            else:
                data[display_name] = runs
                
    return data


def extract_terminal_residuals(algo_runs: dict[str, list[pd.DataFrame]]) -> dict[str, list[float]]:
    """Extract terminal residuals (minimum raw_y achieved per run)."""
    residuals = {}
    for algo, runs in algo_runs.items():
        residuals[algo] = [float(df['raw_y'].min()) for df in runs if not df.empty]
    return residuals

print('Data loading and extraction functions ready.')


Data loading and extraction functions ready.


## 2. Load Benchmark Trajectories & Verify Coverage

In [3]:
all_benchmark_data = {}
summary_records = []

for dim_dir in sorted(IOH_LOGS_DIR.glob('*D')):
    if not dim_dir.is_dir():
        continue
    try:
        dim = int(dim_dir.name.replace('D', ''))
    except ValueError:
        continue
        
    for std_dir in sorted(dim_dir.glob('std_*')):
        if not std_dir.is_dir():
            continue
        try:
            noise_std = float(std_dir.name.replace('std_', ''))
        except ValueError:
            continue
            
        for prob_dir in sorted(std_dir.glob('f*')):
            if not prob_dir.is_dir():
                continue
            try:
                p_id = int(prob_dir.name.replace('f', ''))
            except ValueError:
                continue
                
            algo_runs = load_problem_benchmark_data(prob_dir)
            if algo_runs:
                all_benchmark_data[(dim, noise_std, p_id)] = algo_runs
                for algo, runs in algo_runs.items():
                    summary_records.append({
                        'Dim': f"{dim}D",
                        'Noise Std': noise_std,
                        'Problem': BBOB_NAMES.get(p_id, f'f{p_id}'),
                        'Algorithm': algo,
                        'Runs': len(runs)
                    })

df_coverage = pd.DataFrame(summary_records)
print(f"🎯 Loaded benchmark data for {len(all_benchmark_data)} unique problem configurations.")
if not df_coverage.empty:
    coverage_table = df_coverage.groupby(['Dim', 'Noise Std', 'Problem', 'Algorithm'])['Runs'].sum().unstack(fill_value=0)
    try:
        display(coverage_table)
    except NameError:
        print(coverage_table.to_string())


/var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/ipykernel_43497/1752984722.py:23: DtypeWarning: Columns (0: evaluations, 1: raw_y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(io.StringIO(csv_data), sep=r'\s+')


🎯 Loaded benchmark data for 20 unique problem configurations.


Algorithm                                CMAES  DE  \
Dim Noise Std Problem                                
2D  0.00      Discus (f11)                   1   1   
              Gallagher 101 Peaks (f21)      1   1   
              Rastrigin (f15)                1   1   
              Rosenbrock (f8)                1   1   
              Sphere (f1)                    1   1   
    0.05      Discus (f11)                   1   1   
              Gallagher 101 Peaks (f21)      1   1   
              Rastrigin (f15)                1   1   
              Rosenbrock (f8)                1   1   
              Sphere (f1)                    1   1   
3D  0.00      Discus (f11)                   1   1   
              Gallagher 101 Peaks (f21)      1   1   
              Rastrigin (f15)                1   1   
              Rosenbrock (f8)                1   1   
              Sphere (f1)                    1   1   
    0.05      Discus (f11)                   1   1   
              Gallagher 101 Peaks (f21)      1   1   
              Rastrigin (f15)                1   1   
              Rosenbrock (f8)                1   1   
              Sphere (f1)                    1   1   

Algorithm                                LLAMEA_QWEN2.5-CODER-14B-Q4_K_M  \
Dim Noise Std Problem                                                      
2D  0.00      Discus (f11)                                             0   
              Gallagher 101 Peaks (f21)                                0   
              Rastrigin (f15)                                          0   
              Rosenbrock (f8)                                          0   
              Sphere (f1)                                              0   
    0.05      Discus (f11)                                             0   
              Gallagher 101 Peaks (f21)                                0   
              Rastrigin (f15)                                          0   
              Rosenbrock (f8)                                          0   
              Sphere (f1)                                              0   
3D  0.00      Discus (f11)                                             1   
              Gallagher 101 Peaks (f21)                                1   
              Rastrigin (f15)                                          1   
              Rosenbrock (f8)                                          1   
              Sphere (f1)                                              1   
    0.05      Discus (f11)                                             1   
              Gallagher 101 Peaks (f21)                                1   
              Rastrigin (f15)                                          1   
              Rosenbrock (f8)                                          1   
              Sphere (f1)                                              1   

Algorithm                                LLAMEA_QWEN2.5-CODER-7B-Q4_K_M  \
Dim Noise Std Problem                                                     
2D  0.00      Discus (f11)                                            1   
              Gallagher 101 Peaks (f21)                               1   
              Rastrigin (f15)                                         1   
              Rosenbrock (f8)                                         1   
              Sphere (f1)                                             1   
    0.05      Discus (f11)                                            1   
              Gallagher 101 Peaks (f21)                               1   
              Rastrigin (f15)                                         1   
              Rosenbrock (f8)                                         1   
              Sphere (f1)                                             1   
3D  0.00      Discus (f11)                                            1   
              Gallagher 101 Peaks (f21)                               1   
              Rastrigin (f15)                                         1   
             

## 3. High-Resolution Figures Generation (Strategy-Specific & Comparative Overlays)

Generates:
1. **`{strategy}/f{p_id}.png`**: Focused dual-panel plots comparing a specific prompt strategy against `CMA-ES`, `DE`, `PSO`.
2. **`comparative/f{p_id}_all_solvers.png`**: Comprehensive overlay of all 7 solvers.
3. **`comparative/prompt_ablation_ecdf.png`**: Head-to-head ECDF comparing **only prompt strategies** across all problems.

In [4]:
eval_grid = np.logspace(0, 5, 200)
targets   = np.logspace(-8, 2, 100)

# Group by (dim, noise_std)
grouped_configs = {}
for (dim, noise_std, p_id), algo_runs in all_benchmark_data.items():
    grouped_configs.setdefault((dim, noise_std), {})[p_id] = algo_runs

for (dim, noise_std), problem_dict in grouped_configs.items():
    cond_fig_dir = FIGURES_DIR / f"{dim}D" / f"std_{noise_std}"
    comp_dir = cond_fig_dir / 'comparative'
    comp_dir.mkdir(parents=True, exist_ok=True)
    
    # Discover prompt strategies present in this condition
    strats_present = set()
    for algo_runs in problem_dict.values():
        for a in algo_runs:
            if 'LLaMEA (' in a:
                strat_name = a.replace('LLaMEA (', '').replace(')', '').lower()
                strats_present.add(strat_name)
    if not strats_present:
        strats_present.add('baseline')
        
    # ── 1. Generate Per-Strategy Focused Figures & All-Solvers Figures ─────────
    for p_id, algo_runs in problem_dict.items():
        p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
        p_class = BBOB_CLASSES.get(p_id, 'Optimization')
        
        # A. All-Solvers Comparative Plot
        fig_all = make_subplots(
            rows=1, cols=2,
            subplot_titles=("Mean Convergence Trajectory (±1σ)", "Empirical Cumulative Distribution Function (ECDF)"),
            horizontal_spacing=0.12
        )
        
        for algo_name, runs in algo_runs.items():
            color = SOLVER_COLORS.get(algo_name, '#7F8C8D')
            dash  = SOLVER_DASHES.get(algo_name, 'solid')
            
            # Interpolate convergence
            interp_runs = []
            terminals = []
            for df in runs:
                if df.empty:
                    continue
                interp_y = np.interp(eval_grid, df['evaluations'].values, df['raw_y'].values, left=df['raw_y'].iloc[0], right=df['raw_y'].iloc[-1])
                interp_runs.append(interp_y)
                terminals.append(float(df['raw_y'].min()))
                
            if interp_runs:
                arr = np.array(interp_runs)
                mean_y = np.clip(np.mean(arr, axis=0), 1e-16, 1e10)
                fig_all.add_trace(go.Scatter(
                    x=eval_grid, y=mean_y, mode='lines',
                    name=algo_name, line=dict(color=color, width=2.5 if 'LLaMEA' in algo_name else 1.8, dash=dash)
                ), row=1, col=1)
                
            if terminals:
                ecdf_vals = [np.mean(np.array(terminals) <= t) for t in targets]
                fig_all.add_trace(go.Scatter(
                    x=targets, y=ecdf_vals, mode='lines',
                    name=algo_name, showlegend=False, line=dict(color=color, width=2.5 if 'LLaMEA' in algo_name else 1.8, dash=dash)
                ), row=1, col=2)
                
        fig_all.update_xaxes(type="log", title_text="Evaluations", row=1, col=1)
        fig_all.update_yaxes(type="log", title_text="Best Clean Error", row=1, col=1)
        fig_all.update_xaxes(type="log", title_text="Target Precision (τ)", autorange="reversed", row=1, col=2)
        fig_all.update_yaxes(title_text="Fraction of Solved Trials", range=[-0.05, 1.05], row=1, col=2)
        fig_all.update_layout(
            template="plotly_white", height=450, width=1050,
            title_text=f"<b>{p_name} ({p_class}) — {dim}D (noise_std={noise_std}) — All Solvers</b>",
            legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
        )
        
        all_plot_path = comp_dir / f"f{p_id}_all_solvers.png"
        try:
            fig_all.write_image(str(all_plot_path), scale=2)
        except Exception:
            pass
            
        # B. Focused Per-Strategy Plots
        for strat in strats_present:
            strat_dir = cond_fig_dir / strat
            strat_dir.mkdir(parents=True, exist_ok=True)
            
            strat_label = f"LLaMEA ({strat.capitalize()})"
            focused_algos = ['CMAES', 'DE', 'PSO', strat_label, 'LLaMEA Champion']
            
            fig_focused = make_subplots(
                rows=1, cols=2,
                subplot_titles=(f"{strat.capitalize()} Convergence Trajectory", f"{strat.capitalize()} ECDF"),
                horizontal_spacing=0.12
            )
            
            for algo_name, runs in algo_runs.items():
                if algo_name not in focused_algos:
                    continue
                color = SOLVER_COLORS.get(algo_name, '#7F8C8D')
                dash  = SOLVER_DASHES.get(algo_name, 'solid')
                
                interp_runs = []
                terminals = []
                for df in runs:
                    if df.empty:
                        continue
                    interp_y = np.interp(eval_grid, df['evaluations'].values, df['raw_y'].values, left=df['raw_y'].iloc[0], right=df['raw_y'].iloc[-1])
                    interp_runs.append(interp_y)
                    terminals.append(float(df['raw_y'].min()))
                    
                if interp_runs:
                    arr = np.array(interp_runs)
                    mean_y = np.clip(np.mean(arr, axis=0), 1e-16, 1e10)
                    fig_focused.add_trace(go.Scatter(
                        x=eval_grid, y=mean_y, mode='lines',
                        name=algo_name, line=dict(color=color, width=2.5 if 'LLaMEA' in algo_name else 1.8, dash=dash)
                    ), row=1, col=1)
                    
                if terminals:
                    ecdf_vals = [np.mean(np.array(terminals) <= t) for t in targets]
                    fig_focused.add_trace(go.Scatter(
                        x=targets, y=ecdf_vals, mode='lines',
                        name=algo_name, showlegend=False, line=dict(color=color, width=2.5 if 'LLaMEA' in algo_name else 1.8, dash=dash)
                    ), row=1, col=2)
                    
            fig_focused.update_xaxes(type="log", title_text="Evaluations", row=1, col=1)
            fig_focused.update_yaxes(type="log", title_text="Best Clean Error", row=1, col=1)
            fig_focused.update_xaxes(type="log", title_text="Target Precision (τ)", autorange="reversed", row=1, col=2)
            fig_focused.update_yaxes(title_text="Fraction of Solved Trials", range=[-0.05, 1.05], row=1, col=2)
            fig_focused.update_layout(
                template="plotly_white", height=450, width=1050,
                title_text=f"<b>{p_name} — {dim}D (noise_std={noise_std}) — {strat.capitalize()} vs Baselines</b>",
                legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
            )
            
            focused_plot_path = strat_dir / f"f{p_id}.png"
            try:
                fig_focused.write_image(str(focused_plot_path), scale=2)
            except Exception:
                pass
                
    # ── 2. Multi-Problem Prompt Ablation ECDF ─────────────────────────────────
    fig_ablation = go.Figure()
    for strat in sorted(strats_present):
        strat_label = f"LLaMEA ({strat.capitalize()})"
        all_strat_terminals = []
        for p_id, algo_runs in problem_dict.items():
            if strat_label in algo_runs:
                for df in algo_runs[strat_label]:
                    if not df.empty:
                        all_strat_terminals.append(float(df['raw_y'].min()))
            elif 'LLaMEA Champion' in algo_runs and strat == 'baseline':
                for df in algo_runs['LLaMEA Champion']:
                    if not df.empty:
                        all_strat_terminals.append(float(df['raw_y'].min()))
                        
        if all_strat_terminals:
            ecdf_vals = [np.mean(np.array(all_strat_terminals) <= t) for t in targets]
            color = SOLVER_COLORS.get(strat_label, '#16A085')
            dash  = SOLVER_DASHES.get(strat_label, 'solid')
            fig_ablation.add_trace(go.Scatter(
                x=targets, y=ecdf_vals, mode='lines+markers',
                name=f"{strat.capitalize()} Strategy (N={len(all_strat_terminals)} trials)",
                line=dict(color=color, width=3, dash=dash)
            ))
            
    fig_ablation.update_xaxes(type="log", title_text="Target Precision Threshold (τ)", autorange="reversed")
    fig_ablation.update_yaxes(title_text="Aggregate Proportion of Solved Trials", range=[-0.05, 1.05])
    fig_ablation.update_layout(
        template="plotly_white", height=450, width=900,
        title=f"<b>Prompt Strategy Ablation ECDF ({dim}D, noise_std={noise_std}) Across All Target Problems</b>",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
    )
    ablation_plot_path = comp_dir / 'prompt_ablation_ecdf.png'
    try:
        fig_ablation.write_image(str(ablation_plot_path), scale=2)
    except Exception:
        pass

print('✨ All strategy-specific and comparative figures generated successfully!')


✨ All strategy-specific and comparative figures generated successfully!


## 4. Effect Size Metric: Vargha-Delaney (A12)
Quantifies the probability that algorithm A achieves lower (better) objective values than algorithm B ($A_{12} < 0.5$ favors A).

In [5]:
def vargha_delaney_a12(sample1: list[float], sample2: list[float]) -> tuple[float, str]:
    """Calculates Vargha-Delaney A12 effect size statistic."""
    m = len(sample1)
    n = len(sample2)
    if m == 0 or n == 0:
        return 0.5, 'N/A'
        
    ranks = pd.Series(sample1 + sample2).rank(method='average').values
    r1 = np.sum(ranks[:m])
    a12 = float((r1 - m * (m + 1) / 2) / (m * n))
    
    d = abs(a12 - 0.5)
    if d < 0.06:
        mag = 'Negligible'
    elif d < 0.14:
        mag = 'Small'
    elif d < 0.21:
        mag = 'Medium'
    else:
        mag = 'Large'
        
    return a12, mag

print('Vargha-Delaney A12 calculator ready.')


Vargha-Delaney A12 calculator ready.


## 5. 3-Tier Statistical Hypothesis Testing

Executes:
- **Omnibus Test**: Kruskal-Wallis non-parametric ANOVA across all solvers.
- **Tier 1**: Classical Baselines vs. Classical Baselines (`CMA-ES`, `DE`, `PSO`).
- **Tier 2**: LLaMEA Prompt Strategies vs. Classical Baselines.
- **Tier 3**: Prompt Strategy Ablation (Head-to-Head prompt comparisons).

In [6]:
master_omnibus_results = []
master_pairwise_results = []

for (dim, noise_std, p_id), algo_runs in all_benchmark_data.items():
    p_name  = BBOB_NAMES.get(p_id, f'f{p_id}')
    p_class = BBOB_CLASSES.get(p_id, 'Unknown')
    
    residuals = extract_terminal_residuals(algo_runs)
    valid_algos = [a for a, res in residuals.items() if len(res) >= 1]
    
    # ── 1. Kruskal-Wallis Omnibus Test ────────────────────────────────────────
    if len(valid_algos) >= 2:
        samples = [residuals[a] for a in valid_algos]
        try:
            stat, p_val = kruskal(*samples)
        except Exception:
            stat, p_val = 0.0, 1.0
            
        master_omnibus_results.append({
            'Dim': dim,
            'Noise Std': noise_std,
            'Problem ID': p_id,
            'Problem Name': p_name,
            'Function Class': p_class,
            'Solvers Evaluated': len(valid_algos),
            'H-Statistic': stat,
            'p-value': p_val,
            'Significant Difference': 'Yes' if p_val < 0.05 else 'No'
        })
        
    # Classify solvers into Baselines and LLaMEA Variants
    baselines = [a for a in valid_algos if a in ['CMAES', 'DE', 'PSO']]
    llamea_variants = [a for a in valid_algos if 'LLaMEA' in a or 'LLAMEA' in a]
    
    # ── 2. Tier 1: Classical Baselines Comparison ────────────────────────────
    for i in range(len(baselines)):
        for j in range(i + 1, len(baselines)):
            b1, b2 = baselines[i], baselines[j]
            s1, s2 = residuals[b1], residuals[b2]
            try:
                _, p_val = mannwhitneyu(s1, s2, alternative='two-sided')
            except Exception:
                p_val = 1.0
            a12, mag = vargha_delaney_a12(s1, s2)
            med1, med2 = np.median(s1), np.median(s2)
            
            if med1 == med2:
                outcome = 'Exact Tie'
            elif p_val < 0.05:
                outcome = f"{b1} Wins (Sig)" if a12 < 0.5 else f"{b2} Wins (Sig)"
            else:
                outcome = f"{b1} Ahead (Non-Sig)" if a12 < 0.5 else f"{b2} Ahead (Non-Sig)"
                
            master_pairwise_results.append({
                'Dim': dim,
                'Noise Std': noise_std,
                'Problem ID': p_id,
                'Problem Name': p_name,
                'Function Class': p_class,
                'Comparison Tier': 'Tier 1: Classical Baselines',
                'Solver 1': b1,
                'Solver 2': b2,
                'Solver 1 Med Res': med1,
                'Solver 2 Med Res': med2,
                'Mann-Whitney p-val': p_val,
                'A12 Effect Size': a12,
                'Effect Magnitude': mag,
                'Outcome': outcome
            })
            
    # ── 3. Tier 2: LLaMEA Prompt Strategies vs. Classical Baselines ──────────
    for llm_algo in llamea_variants:
        s_llm = residuals[llm_algo]
        for b in baselines:
            s_base = residuals[b]
            try:
                _, p_val = mannwhitneyu(s_llm, s_base, alternative='two-sided')
            except Exception:
                p_val = 1.0
            a12, mag = vargha_delaney_a12(s_llm, s_base)
            med_llm, med_base = np.median(s_llm), np.median(s_base)
            
            if med_llm == med_base:
                outcome = 'Exact Tie'
            elif p_val < 0.05:
                outcome = 'LLaMEA Wins (Sig)' if a12 < 0.5 else f"{b} Wins (Sig)"
            else:
                outcome = 'LLaMEA Ahead (Non-Sig)' if a12 < 0.5 else f"{b} Ahead (Non-Sig)"
                
            master_pairwise_results.append({
                'Dim': dim,
                'Noise Std': noise_std,
                'Problem ID': p_id,
                'Problem Name': p_name,
                'Function Class': p_class,
                'Comparison Tier': 'Tier 2: LLaMEA vs Baselines',
                'Solver 1': llm_algo,
                'Solver 2': b,
                'Solver 1 Med Res': med_llm,
                'Solver 2 Med Res': med_base,
                'Mann-Whitney p-val': p_val,
                'A12 Effect Size': a12,
                'Effect Magnitude': mag,
                'Outcome': outcome
            })
            
    # ── 4. Tier 3: Prompt Strategy Ablation Matrix ───────────────────────────
    for i in range(len(llamea_variants)):
        for j in range(i + 1, len(llamea_variants)):
            l1, l2 = llamea_variants[i], llamea_variants[j]
            s1, s2 = residuals[l1], residuals[l2]
            try:
                _, p_val = mannwhitneyu(s1, s2, alternative='two-sided')
            except Exception:
                p_val = 1.0
            a12, mag = vargha_delaney_a12(s1, s2)
            med1, med2 = np.median(s1), np.median(s2)
            
            if med1 == med2:
                outcome = 'Exact Tie'
            elif p_val < 0.05:
                outcome = f"{l1} Wins (Sig)" if a12 < 0.5 else f"{l2} Wins (Sig)"
            else:
                outcome = f"{l1} Ahead (Non-Sig)" if a12 < 0.5 else f"{l2} Ahead (Non-Sig)"
                
            master_pairwise_results.append({
                'Dim': dim,
                'Noise Std': noise_std,
                'Problem ID': p_id,
                'Problem Name': p_name,
                'Function Class': p_class,
                'Comparison Tier': 'Tier 3: Prompt Ablation',
                'Solver 1': l1,
                'Solver 2': l2,
                'Solver 1 Med Res': med1,
                'Solver 2 Med Res': med2,
                'Mann-Whitney p-val': p_val,
                'A12 Effect Size': a12,
                'Effect Magnitude': mag,
                'Outcome': outcome
            })

df_all_pairwise = pd.DataFrame(master_pairwise_results)
df_all_omnibus  = pd.DataFrame(master_omnibus_results)

print(f"🎯 Evaluated {len(df_all_omnibus)} omnibus tests and {len(df_all_pairwise)} 3-tier pairwise comparisons.")


🎯 Evaluated 20 omnibus tests and 429 3-tier pairwise comparisons.


/Users/nicolaibrahim/Desktop/proj/AAD_LLM/.venv/lib/python3.13/site-packages/scipy/stats/_stats_py.py:8674: RuntimeWarning: invalid value encountered in scalar divide
  h /= ties


## 6. Multi-Format Report Export (Markdown & CSVs)

Exports:
1. **Per-condition folders** in `results/reports/{dim}D/std_{noise_std}/`:
   - `statistical_summary.md` (Formatted 4-section report with visual badges)
   - `statistical_summary.csv` (Complete pairwise tests)
   - `prompt_ablation_summary.csv` (Tier 3 Prompt comparisons)
   - `baselines_comparison_summary.csv` (Tier 1 Baseline comparisons)
   - `kruskal_wallis_summary.csv` (Omnibus ANOVA)
2. **Master Consolidated Roll-up** in `results/reports/`.

In [7]:
if not df_all_pairwise.empty:
    # ── 1. Export Condition-Specific Reports ──────────────────────────────────
    grouped = df_all_pairwise.groupby(['Dim', 'Noise Std'])
    
    for (dim, noise_std), df_subset in grouped:
        subset_report_dir = REPORTS_DIR / f"{dim}D" / f"std_{noise_std}"
        subset_report_dir.mkdir(parents=True, exist_ok=True)
        
        # CSV Exports
        df_subset.to_csv(subset_report_dir / 'statistical_summary.csv', index=False)
        df_subset[df_subset['Comparison Tier'] == 'Tier 3: Prompt Ablation'].to_csv(subset_report_dir / 'prompt_ablation_summary.csv', index=False)
        df_subset[df_subset['Comparison Tier'] == 'Tier 1: Classical Baselines'].to_csv(subset_report_dir / 'baselines_comparison_summary.csv', index=False)
        
        df_omnibus_subset = df_all_omnibus[(df_all_omnibus['Dim'] == dim) & (df_all_omnibus['Noise Std'] == noise_std)]
        df_omnibus_subset.to_csv(subset_report_dir / 'kruskal_wallis_summary.csv', index=False)
        
        # Condition Markdown Report
        noise_title = "Clean Mode (std = 0.0)" if noise_std == 0.0 else f"Noisy Mode (std = {noise_std})"
        tier2 = df_subset[df_subset['Comparison Tier'] == 'Tier 2: LLaMEA vs Baselines']
        llm_wins = len(tier2[tier2['Outcome'].str.contains('LLaMEA Wins', regex=False)])
        base_wins = len(tier2[tier2['Outcome'].str.contains('Wins (Sig)', regex=False) & ~tier2['Outcome'].str.contains('LLaMEA', regex=False)])
        ties = len(tier2) - llm_wins - base_wins
        
        lines = [
            f"# 📊 Benchmark Statistical Summary: {dim}D ({noise_title})",
            "",
            f"> **Target Dimension:** `{dim}D` | **Noise Level:** `{noise_std}`",
            "",
            "## 🏆 1. Win-Loss Summary (LLaMEA vs. Classical Baselines)",
            f"- **Total Comparisons against Baselines:** `{len(tier2)}`",
            rf"- **🟢 LLaMEA Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} < 0.5$):** **`{llm_wins}`** ({llm_wins/max(1, len(tier2))*100:.1f}%)",
            rf"- **🔴 Classical Baselines Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} > 0.5$):** **`{base_wins}`** ({base_wins/max(1, len(tier2))*100:.1f}%)",
            f"- **⚪ Non-Significant Differences & Exact Ties:** **`{ties}`** ({ties/max(1, len(tier2))*100:.1f}%)",
            "",
            "---",
            "## 🌐 2. Omnibus Kruskal-Wallis H-Test (Group Differences Across All Solvers)",
            "",
            "| Problem | Function Class | Solvers | H-Statistic | p-value | Significant Difference? |",
            "| :--- | :--- | :---: | :---: | :---: | :---: |"
        ]
        
        for _, r in df_omnibus_subset.iterrows():
            diff_badge = "🟢 **Yes**" if r['Significant Difference'] == 'Yes' else "⚪ No"
            lines.append(f"| `{r['Problem Name']}` | {r['Function Class']} | {r['Solvers Evaluated']} | {r['H-Statistic']:.3f} | {r['p-value']:.2e} | {diff_badge} |")
            
        # Section 3: LLaMEA vs Baselines
        lines.extend([
            "",
            "---",
            "## 🔬 3. Pairwise Comparisons (LLaMEA Prompt Strategies vs. Classical Baselines)",
            "",
            "| Problem | Function Class | LLaMEA Strategy | Baseline | LLaMEA Median | Baseline Median | MW p-val | A12 Effect Size | Magnitude | Outcome |",
            "| :--- | :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :--- |"
        ])
        
        prev_prob = None
        for _, r in tier2.iterrows():
            prob_str = f"**{r['Problem Name']}**" if r['Problem Name'] != prev_prob else ""
            class_str = r['Function Class'] if r['Problem Name'] != prev_prob else ""
            prev_prob = r['Problem Name']
            
            outcome_str = r['Outcome']
            badge = f"🟢 **{outcome_str}**" if 'LLaMEA Wins' in outcome_str else (f"🔴 **{outcome_str}**" if 'Wins (Sig)' in outcome_str else f"⚪ {outcome_str}")
            lines.append(f"| {prob_str} | {class_str} | **{r['Solver 1']}** | **{r['Solver 2']}** | {r['Solver 1 Med Res']:.2e} | {r['Solver 2 Med Res']:.2e} | {r['Mann-Whitney p-val']:.2e} | {r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |")
            
        # Section 4: Prompt Strategy Ablation Matrix
        tier3 = df_subset[df_subset['Comparison Tier'] == 'Tier 3: Prompt Ablation']
        if not tier3.empty:
            lines.extend([
                "",
                "---",
                "## 🧠 4. Prompt Strategy Ablation Matrix (Head-to-Head Comparisons)",
                "",
                "| Problem | Prompt Strategy A | Prompt Strategy B | Strategy A Median | Strategy B Median | MW p-val | A12 (A < B) | Magnitude | Outcome |",
                "| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :--- |"
            ])
            prev_prob = None
            for _, r in tier3.iterrows():
                prob_str = f"**{r['Problem Name']}**" if r['Problem Name'] != prev_prob else ""
                prev_prob = r['Problem Name']
                outcome_str = r['Outcome']
                badge = f"🟢 **{outcome_str}**" if 'Wins (Sig)' in outcome_str else f"⚪ {outcome_str}"
                lines.append(f"| {prob_str} | **{r['Solver 1']}** | **{r['Solver 2']}** | {r['Solver 1 Med Res']:.2e} | {r['Solver 2 Med Res']:.2e} | {r['Mann-Whitney p-val']:.2e} | {r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |")
                
        # Section 5: Classical Baselines Comparison
        tier1 = df_subset[df_subset['Comparison Tier'] == 'Tier 1: Classical Baselines']
        if not tier1.empty:
            lines.extend([
                "",
                "---",
                "## ⚔️ 5. Classical Baselines Inter-Comparison (CMA-ES vs. DE vs. PSO)",
                "",
                "| Problem | Baseline A | Baseline B | Median A | Median B | MW p-val | A12 (A < B) | Magnitude | Outcome |",
                "| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :--- |"
            ])
            prev_prob = None
            for _, r in tier1.iterrows():
                prob_str = f"**{r['Problem Name']}**" if r['Problem Name'] != prev_prob else ""
                prev_prob = r['Problem Name']
                outcome_str = r['Outcome']
                badge = f"🟢 **{outcome_str}**" if 'Wins (Sig)' in outcome_str else f"⚪ {outcome_str}"
                lines.append(f"| {prob_str} | **{r['Solver 1']}** | **{r['Solver 2']}** | {r['Solver 1 Med Res']:.2e} | {r['Solver 2 Med Res']:.2e} | {r['Mann-Whitney p-val']:.2e} | {r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |")
                
        md_out_path = subset_report_dir / 'statistical_summary.md'
        md_out_path.write_text('\n'.join(lines), encoding='utf-8')
        print(f"  📄 Exported condition report: {md_out_path.relative_to(PROJECT_ROOT)}")
        
    # ── 2. Export Master Consolidated Reports ─────────────────────────────────
    df_all_pairwise.to_csv(REPORTS_DIR / 'statistical_summary.csv', index=False)
    df_all_pairwise[df_all_pairwise['Comparison Tier'] == 'Tier 3: Prompt Ablation'].to_csv(REPORTS_DIR / 'prompt_ablation_summary.csv', index=False)
    df_all_pairwise[df_all_pairwise['Comparison Tier'] == 'Tier 1: Classical Baselines'].to_csv(REPORTS_DIR / 'baselines_comparison_summary.csv', index=False)
    df_all_omnibus.to_csv(REPORTS_DIR / 'kruskal_wallis_summary.csv', index=False)
    
    # Master Markdown Roll-up
    tier2_all = df_all_pairwise[df_all_pairwise['Comparison Tier'] == 'Tier 2: LLaMEA vs Baselines']
    total_llm_wins = len(tier2_all[tier2_all['Outcome'].str.contains('LLaMEA Wins', regex=False)])
    total_base_wins = len(tier2_all[tier2_all['Outcome'].str.contains('Wins (Sig)', regex=False) & ~tier2_all['Outcome'].str.contains('LLaMEA', regex=False)])
    total_ties = len(tier2_all) - total_llm_wins - total_base_wins
    
    master_lines = [
        "# 📊 Comprehensive Benchmark & Statistical Analysis Report",
        "",
        "> **Master Roll-Up Report** evaluating **LLaMEA Prompt Strategies** against classical optimizers (**CMA-ES**, **DE**, **PSO**).",
        "",
        "## 🏆 1. Executive Summary & Win-Loss Metrics (LLaMEA vs. Classical Baselines)",
        f"- **Total Pairwise Tests ($N$):** `{len(tier2_all)}`",
        rf"- **🟢 LLaMEA Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} < 0.5$):** **`{total_llm_wins}`** ({total_llm_wins/max(1, len(tier2_all))*100:.1f}%)",
        rf"- **🔴 Classical Baselines Statistically Significant Wins ($p < 0.05, \hat{{A}}_{{12}} > 0.5$):** **`{total_base_wins}`** ({total_base_wins/max(1, len(tier2_all))*100:.1f}%)",
        f"- **⚪ Non-Significant Differences & Exact Ties:** **`{total_ties}`** ({total_ties/max(1, len(tier2_all))*100:.1f}%)",
        "",
        "---",
        "## 🌐 2. Omnibus Kruskal-Wallis H-Test (Group Differences Across All Solvers)",
        "",
        "| Dim | Noise Std | Problem | Function Class | Solvers | H-Statistic | p-value | Significant Difference? |",
        "| :---: | :---: | :--- | :--- | :---: | :---: | :---: | :---: |"
    ]
    for _, r in df_all_omnibus.iterrows():
        diff_badge = "🟢 **Yes**" if r['Significant Difference'] == 'Yes' else "⚪ No"
        master_lines.append(f"| `{r['Dim']}D` | `{r['Noise Std']}` | `{r['Problem Name']}` | {r['Function Class']} | {r['Solvers Evaluated']} | {r['H-Statistic']:.3f} | {r['p-value']:.2e} | {diff_badge} |")
        
    master_lines.extend([
        "",
        "---",
        "## 🔬 3. Complete Pairwise Evaluation Matrix (LLaMEA Prompt Strategies vs. Baselines)",
        "",
        "| Dim | Noise Std | Problem | Function Class | LLaMEA Strategy | Baseline | LLaMEA Median | Baseline Median | MW p-val | A12 Effect Size | Magnitude | Outcome |",
        "| :---: | :---: | :--- | :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :--- |"
    ])
    
    prev_block = None
    for _, r in tier2_all.iterrows():
        block_key = (r['Dim'], r['Noise Std'], r['Problem Name'])
        if block_key != prev_block:
            dim_str = f"`{r['Dim']}D`"
            noise_str = f"`{r['Noise Std']}`"
            prob_str = f"**{r['Problem Name']}**"
            class_str = r['Function Class']
            prev_block = block_key
        else:
            dim_str, noise_str, prob_str, class_str = "", "", "", ""
            
        outcome_str = r['Outcome']
        badge = f"🟢 **{outcome_str}**" if 'LLaMEA Wins' in outcome_str else (f"🔴 **{outcome_str}**" if 'Wins (Sig)' in outcome_str else f"⚪ {outcome_str}")
        master_lines.append(f"| {dim_str} | {noise_str} | {prob_str} | {class_str} | **{r['Solver 1']}** | **{r['Solver 2']}** | {r['Solver 1 Med Res']:.2e} | {r['Solver 2 Med Res']:.2e} | {r['Mann-Whitney p-val']:.2e} | {r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |")
        
    tier3_all = df_all_pairwise[df_all_pairwise['Comparison Tier'] == 'Tier 3: Prompt Ablation']
    if not tier3_all.empty:
        master_lines.extend([
            "",
            "---",
            "## 🧠 4. Prompt Strategy Ablation Matrix (Head-to-Head Comparisons)",
            "",
            "| Dim | Noise Std | Problem | Prompt Strategy A | Prompt Strategy B | Strategy A Median | Strategy B Median | MW p-val | A12 (A < B) | Magnitude | Outcome |",
            "| :---: | :---: | :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |"
        ])
        prev_block = None
        for _, r in tier3_all.iterrows():
            block_key = (r['Dim'], r['Noise Std'], r['Problem Name'])
            if block_key != prev_block:
                dim_str = f"`{r['Dim']}D`"
                noise_str = f"`{r['Noise Std']}`"
                prob_str = f"**{r['Problem Name']}**"
                prev_block = block_key
            else:
                dim_str, noise_str, prob_str = "", "", ""
                
            outcome_str = r['Outcome']
            badge = f"🟢 **{outcome_str}**" if 'Wins (Sig)' in outcome_str else f"⚪ {outcome_str}"
            master_lines.append(f"| {dim_str} | {noise_str} | {prob_str} | **{r['Solver 1']}** | **{r['Solver 2']}** | {r['Solver 1 Med Res']:.2e} | {r['Solver 2 Med Res']:.2e} | {r['Mann-Whitney p-val']:.2e} | {r['A12 Effect Size']:.3f} | {r['Effect Magnitude']} | {badge} |")
            
    master_md_path = REPORTS_DIR / 'statistical_summary.md'
    master_md_path.write_text('\n'.join(master_lines), encoding='utf-8')
    print(f"\n✨ Master consolidated report exported to: {master_md_path.relative_to(PROJECT_ROOT)}")


  📄 Exported condition report: results/reports/2D/std_0.0/statistical_summary.md
  📄 Exported condition report: results/reports/2D/std_0.05/statistical_summary.md
  📄 Exported condition report: results/reports/3D/std_0.0/statistical_summary.md
  📄 Exported condition report: results/reports/3D/std_0.05/statistical_summary.md

✨ Master consolidated report exported to: results/reports/statistical_summary.md
